# Bearing fault diagnosis — Colab run

Runs the data pipeline: install, tests, download the Paderborn data, check the loader, build the feature table.

**Before you start:** upload the project folder (`load_bearing_research_paper`) to Google Drive, e.g. `MyDrive/load_bearing_research_paper`.

- Raw data (~4.6 GB download) goes to Colab's temporary disk and is erased when the session ends.
- The feature table is saved to the project folder in Drive, so later steps don't need the raw data.

## 1. Settings

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/load_bearing_research_paper"  # change if the folder is elsewhere in Drive
RAW_DIR = "/content/paderborn_raw"  # Colab local disk (temporary)

## 2. Mount Drive and open the project

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.chdir(PROJECT_DIR)
os.environ["BEARING_RAW_DIR"] = RAW_DIR
print(os.getcwd())

## 3. Install dependencies and run the tests

In [ ]:
!apt-get -qq install -y unrar > /dev/null
!pip install -q -r requirements.txt
!python -m pytest -q

## 4. Download the data

Downloads the 29 study bearings (~160 MB each) and keeps only the `N15_M07_F10` recordings.
Takes roughly 10–20 minutes. Re-running skips bearings that are already present.

Data licence: CC BY-NC 4.0 — cite Lessmeier et al. (2016) and the Paderborn KAt-DataCenter.

In [ ]:
!python scripts/download_paderborn.py

## 5. Check the loader on a real file

Confirms the `.mat` layout: the channel list should include `vibration_1`, and each recording should be about 4 s (≈256,000 samples).

In [ ]:
import sys
sys.path.insert(0, "src")

from scipy.io import loadmat
from bearing_uq.data.files import recording_path
from bearing_uq.data.loader import _channel_name, load_vibration

path = recording_path("KA01", 1, root=RAW_DIR)
record = loadmat(path)[path.stem][0, 0]
print("Channels:", [_channel_name(ch) for ch in record["Y"].ravel()])

x = load_vibration(path)
print(f"{len(x)} samples = {len(x) / 64000:.2f} s")

## 6. Build the feature table (saved to Drive)

In [ ]:
!python scripts/build_features.py

## 7. Sanity check

Bearings per group, and median fault-frequency features. Outer-race bearings should show higher `bpfo_h1`,
inner-race bearings higher `bpfi_h1`. If not, the envelope band or fault frequencies need checking before modelling.

In [ ]:
import pandas as pd

df = pd.read_csv("data/features/features_N15_M07_F10.csv")
print(df.groupby(["origin", "label"]).bearing.nunique(), "\n")
df.groupby(["origin", "label"])[["kurtosis", "bpfo_h1", "bpfi_h1"]].median().round(2)